# Exploratory Data Analysis & Visualisation: Sri Lanka Power Management Study

**Lead Author & Role:** Senior Data Scientist & Power Systems Engineer  
**Study Scope:** January 27, 2026 – July 26, 2026 (181 Calendar Days / ~6 Months)  
**Target Platform:** Google Colab / Local Jupyter Environment  
**Dataset Directory:** `Cleaned Data/` (`Load Curve`, `Energy Data`, `Night Peak Data`, `Peak Data`, `Reservoir Data`)

---

## 1. Project Context & Engineering Objectives

This study investigates power system dispatch, load profiles, hydrological storage, and renewable integration on the Sri Lankan national grid (Ceylon Electricity Board - CEB) to evaluate three strategic aims:

1. **Back Pumping (Pumped Storage Hydropower - PSH):**  
   Utilising excess electrical energy (during low-demand off-peak hours or variable renewable surges) to pump water from lower reservoirs to upper reservoirs, maintaining grid balance, stabilizing frequency, and storing energy for dispatch during peak hours.
2. **Solar-Excess Back Pumping:**  
   Identifying midday solar generation surplus (or potential overgeneration beyond base-load minimums) to drive pumping motors, directly converting midday solar energy into stored potential hydro energy for discharge during the domestic evening peak (18:30–20:00).
3. **Fossil Fuel Reduction:**  
   Quantifying thermal energy dispatch (Lakvijaya Coal and Heavy Fuel Oil/Diesel thermal peakers) to determine where renewables and pumped hydro can systematically displace expensive, carbon-intensive generation.

---

## 2. Dual-Lens Analytical Framework

Every analytical section, summary table, and visualization is strictly classified under one of two operational lenses:

- **`[GENERAL]` Lens:** Objective baseline characteristics of the Sri Lankan power system regardless of specific project interventions (e.g., total demand envelope, seasonal monsoon shifts, reservoir depletion trends, standard load durations).
- **`[PROJECT-BIASED]` Lens:** Analyses designed specifically to test hypotheses surrounding the three project aims (e.g., solar surplus magnitude, midday duck-curve depth, night-peak thermal share, pumping energy headroom, and merit-order thermal displacement).

---

## 3. Strict Methodological & Working Rules

- **Terminology Precision:**  
  - **Generation / Energy:** Refers strictly to energy in **Gigawatt-hours (GWh)** (derived from daily energy records or integrated load curves).  
  - **Power / Demand:** Refers strictly to instantaneous active power dispatch in **Megawatts (MW)** (derived from 15-minute load curves and peak snapshots).
- **Data Integrity & Missing Data:**  
  - The dataset spans 181 calendar days. `Load Curve`, `Night Peak Data`, and `Reservoir Data` contain complete data for all 181 days.  
  - `Energy Data` and `Peak Data` have an upstream blank file on **`2026-03-01`**. In line with engineering standards, this missing date is **never silently altered**. Linear interpolation is applied solely for continuous time-series plotting and is explicitly flagged (`is_interpolated = True`).
- **Evidence vs. Scenarios:**  
  - All metrics, charts, and summary tables present **empirical evidence** directly measured from the grid.  
  - What-if calculations or storage assumptions are strictly isolated and labeled as **scenarios**.


In [ ]:
# =============================================================================
# ENVIRONMENT SETUP, PATH CONFIGURATION, AND PALETTES
# =============================================================================
import os
import sys
import glob
import json
import warnings
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# 1. Automated Path Discovery (Supports Google Colab and Local Workspaces)
# -----------------------------------------------------------------------------
CANDIDATE_PATHS = [
    "./Cleaned Data",
    "../Cleaned Data",
    "d:/GiTHuB/Power-Management-Study/Cleaned Data",
    "/content/Cleaned Data",
    "/content/Power-Management-Study/Cleaned Data"
]

DATA_DIR = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if DATA_DIR is None:
    print("[INFO] Cleaned Data not found in standard directories.")
    print("If running in Google Colab, executing automatic repository clone...")
    try:
        import subprocess
        subprocess.run(["git", "clone", "https://github.com/Sasindu99-ai/Power-Management-Study.git"], check=True)
        DATA_DIR = "/content/Power-Management-Study/Cleaned Data"
    except Exception as e:
        print(f"[ERROR] Could not auto-clone repository: {e}")
        print("Please upload the 'Cleaned Data' folder to your Colab session.")
else:
    print(f"[SUCCESS] Dataset root confirmed: '{DATA_DIR}'")

# -----------------------------------------------------------------------------
# 2. Typography & Matplotlib Engineering Styling
# -----------------------------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 110,
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
    'axes.edgecolor': '#475569',
    'axes.linewidth': 1.0
})

# -----------------------------------------------------------------------------
# 3. Dedicated Engineering Palettes (Matching Reference Design)
# -----------------------------------------------------------------------------
COLOR_FOSSIL = '#6B1D2F'     # Deep Maroon / Burgundy (Fossil Fuel: Coal + Oil)
COLOR_RENEWABLE = '#2E8B57'  # Forest Green (Renewables: Hydro + Solar + Wind + Biomass)
COLOR_TOTAL = '#8C827A'      # Muted Taupe / Neutral (Total Demand)

COLOR_COAL = '#4E342E'       # Dark Earth Brown
COLOR_HYDRO = '#1E88E5'      # Cobalt Royal Blue
COLOR_THERMAL = '#D32F2F'    # Crimson / Firebrick
COLOR_WIND = '#00897B'       # Deep Teal
COLOR_SOLAR = '#F59E0B'      # Amber Gold
COLOR_MINIHYDRO = '#26A69A'  # Light Teal
COLOR_BIOMASS = '#8D6E63'    # Warm Tan

SOURCE_PALETTE = {
    'Coal': COLOR_COAL,
    'Major Hydro': COLOR_HYDRO,
    'Thermal-Oil': COLOR_THERMAL,
    'Wind': COLOR_WIND,
    'Solar': COLOR_SOLAR,
    'SPP Minihydro': COLOR_MINIHYDRO,
    'SPP Biomass': COLOR_BIOMASS
}

REGIME_PALETTE = {
    'Day Peak': '#F59E0B',
    'Night Peak': '#DC2626',
    'Off Peak': '#3B82F6'
}

print("[SUCCESS] Styling and palettes initialized.")


In [ ]:
# =============================================================================
# DATA INGESTION, PREPROCESSING, AND MASTER CONSOLIDATION PIPELINE
# =============================================================================
print("[1/5] Loading 15-Minute Load Curve datasets...")
lc_files = sorted(glob.glob(os.path.join(DATA_DIR, "Load Curve", "*.csv")))
df_lc_list = [pd.read_csv(f) for f in lc_files]
df_lc = pd.concat(df_lc_list, ignore_index=True)
df_lc['DateTime'] = pd.to_datetime(df_lc['DateTime'])
df_lc = df_lc.sort_values('DateTime').reset_index(drop=True)

# Derive Engineering Metrics
SOURCES_MW = ['Coal', 'Major Hydro', 'Thermal-Oil', 'Wind', 'Solar', 'SPP Minihydro', 'SPP Biomass']
RENEWABLES_MW = ['Major Hydro', 'Wind', 'Solar', 'SPP Minihydro', 'SPP Biomass']
FOSSIL_MW = ['Coal', 'Thermal-Oil']

df_lc['Total_MW'] = df_lc[SOURCES_MW].sum(axis=1)
df_lc['Renewable_MW'] = df_lc[RENEWABLES_MW].sum(axis=1)
df_lc['Fossil_MW'] = df_lc[FOSSIL_MW].sum(axis=1)
df_lc['Net_Load_MW'] = df_lc['Total_MW'] - (df_lc['Solar'] + df_lc['Wind'])
df_lc['Renewable_Share_Pct'] = (df_lc['Renewable_MW'] / df_lc['Total_MW']) * 100.0
df_lc['Fossil_Share_Pct'] = (df_lc['Fossil_MW'] / df_lc['Total_MW']) * 100.0

df_lc['Date'] = df_lc['DateTime'].dt.date
df_lc['Time'] = df_lc['DateTime'].dt.time
df_lc['Time_Str'] = df_lc['DateTime'].dt.strftime('%H:%M:%S')
df_lc['DayOfWeek'] = df_lc['DateTime'].dt.day_name()
df_lc['DayOfWeek_Num'] = df_lc['DateTime'].dt.dayofweek # 0=Mon, 6=Sun
df_lc['IsWeekend'] = df_lc['DayOfWeek_Num'].isin([5, 6])
df_lc['Month_Name'] = df_lc['DateTime'].dt.month_name()

print(f" -> Load Curve loaded: {len(df_lc):,} records across {df_lc['Date'].nunique()} days.")

# -----------------------------------------------------------------------------
# 2. Loading Daily Energy Generation Datasets (GWh)
# -----------------------------------------------------------------------------
print("[2/5] Loading Daily Energy Data...")
ed_files = sorted(glob.glob(os.path.join(DATA_DIR, "Energy Data", "*.csv")))
df_ed = pd.concat([pd.read_csv(f) for f in ed_files], ignore_index=True)
df_ed['Date'] = pd.to_datetime(df_ed['dateTime']).dt.date
df_ed = df_ed.drop_duplicates(subset=['Date', 'stationGroupName']).reset_index(drop=True)

# Pivot by Station Group
df_ed_pivot = df_ed.pivot(index='Date', columns='stationGroupName', values='energyGWh').fillna(0)

# Categorize Energy by Fuel Source
HYDRO_COMPLEXES = ['Laxapana Complex', 'Mahaweli Complex', 'Samanala Complex']
RENEWABLE_ENERGY_GROUPS = HYDRO_COMPLEXES + ['CEB Wind', 'IPP Wind', 'IPP Solar', 'IPP BMP', 'IPP MHP']
FOSSIL_ENERGY_GROUPS = ['CEB Thermal - Coal', 'CEB Thermal - Oil', 'IPP Thermal']

df_ed_pivot['Renewable_GWh'] = df_ed_pivot[[col for col in RENEWABLE_ENERGY_GROUPS if col in df_ed_pivot.columns]].sum(axis=1)
df_ed_pivot['Fossil_GWh'] = df_ed_pivot[[col for col in FOSSIL_ENERGY_GROUPS if col in df_ed_pivot.columns]].sum(axis=1)
df_ed_pivot['Total_GWh'] = df_ed_pivot['Renewable_GWh'] + df_ed_pivot['Fossil_GWh']
df_ed_pivot['Renewable_Energy_Share_Pct'] = (df_ed_pivot['Renewable_GWh'] / df_ed_pivot['Total_GWh']) * 100.0
df_ed_pivot['Fossil_Energy_Share_Pct'] = (df_ed_pivot['Fossil_GWh'] / df_ed_pivot['Total_GWh']) * 100.0

# Transparent Handling of Missing Date 2026-03-01
all_dates = pd.date_range(start=df_lc['Date'].min(), end=df_lc['Date'].max(), freq='D').date
df_ed_full = df_ed_pivot.reindex(all_dates)
df_ed_full['is_interpolated'] = df_ed_full['Total_GWh'].isna()
df_ed_full = df_ed_full.interpolate(method='linear')
print(f" -> Energy Data loaded: {len(df_ed_pivot)} empirical days + 1 interpolated day (2026-03-01).")

# -----------------------------------------------------------------------------
# 3. Loading Peak Data (Day Peak, Night Peak, Off Peak MW & Mvar)
# -----------------------------------------------------------------------------
print("[3/5] Loading Peak Regime snapshots...")
pd_files = sorted(glob.glob(os.path.join(DATA_DIR, "Peak Data", "*.csv")))
df_pd_raw = pd.concat([pd.read_csv(f) for f in pd_files], ignore_index=True)
df_pd_raw['Date'] = pd.to_datetime(df_pd_raw['dateTime']).dt.date
df_pd_raw['Time'] = pd.to_datetime(df_pd_raw['dateTime']).dt.time

# De-duplicate peak regimes per date (take maximum active MW per regime)
df_pd_clean = df_pd_raw.sort_values('activePowerMW', ascending=False).drop_duplicates(subset=['Date', 'stationGroupName']).reset_index(drop=True)

df_peak_mw = df_pd_clean.pivot(index='Date', columns='stationGroupName', values='activePowerMW')
df_peak_mvar = df_pd_clean.pivot(index='Date', columns='stationGroupName', values='reactivePowerMvar')
df_peak_time = df_pd_clean.pivot(index='Date', columns='stationGroupName', values='dateTime')

# Reindex and interpolate missing 2026-03-01
df_peak_mw = df_peak_mw.reindex(all_dates).interpolate(method='linear')
df_peak_mvar = df_peak_mvar.reindex(all_dates).interpolate(method='linear')
print(f" -> Peak Data loaded: 180 empirical days + 1 interpolated day (2026-03-01).")

# -----------------------------------------------------------------------------
# 4. Loading Night Peak Data (19:00:00 Dispatch by Station Group)
# -----------------------------------------------------------------------------
print("[4/5] Loading Night Peak Data...")
npd_files = sorted(glob.glob(os.path.join(DATA_DIR, "Night Peak Data", "*.csv")))
df_npd = pd.concat([pd.read_csv(f) for f in npd_files], ignore_index=True)
df_npd['Date'] = pd.to_datetime(df_npd['dateTime']).dt.date
df_npd = df_npd.drop_duplicates(subset=['Date', 'stationGroupName']).reset_index(drop=True)

df_np_mw = df_npd.pivot(index='Date', columns='stationGroupName', values='activePowerMW').fillna(0)

FOSSIL_NP_GROUPS = ['CEB Thermal - Coal', 'CEB Thermal - Oil', 'IPP Thermal']
RENEWABLE_NP_GROUPS = [c for c in df_np_mw.columns if c not in FOSSIL_NP_GROUPS and c != 'Capacitors']

df_np_mw['Fossil_NP_MW'] = df_np_mw[[c for c in FOSSIL_NP_GROUPS if c in df_np_mw.columns]].sum(axis=1)
df_np_mw['Renewable_NP_MW'] = df_np_mw[[c for c in RENEWABLE_NP_GROUPS if c in df_np_mw.columns]].sum(axis=1)
df_np_mw['Total_NP_MW'] = df_np_mw['Fossil_NP_MW'] + df_np_mw['Renewable_NP_MW']
df_np_mw['Fossil_NP_Share_Pct'] = (df_np_mw['Fossil_NP_MW'] / df_np_mw['Total_NP_MW']) * 100.0
df_np_mw['Renewable_NP_Share_Pct'] = (df_np_mw['Renewable_NP_MW'] / df_np_mw['Total_NP_MW']) * 100.0
print(f" -> Night Peak Data loaded: {len(df_np_mw)} days (100% complete).")

# -----------------------------------------------------------------------------
# 5. Loading Reservoir Hydrological Records
# -----------------------------------------------------------------------------
print("[5/5] Loading Reservoir Storage and Rainfall...")
rd_files = sorted(glob.glob(os.path.join(DATA_DIR, "Reservoir Data", "*.csv")))
df_rd = pd.concat([pd.read_csv(f) for f in rd_files], ignore_index=True)
df_rd['Date'] = pd.to_datetime(df_rd['dateTime']).dt.date
df_rd = df_rd.drop_duplicates(subset=['Date', 'reservoirName']).reset_index(drop=True)

df_res_storage = df_rd.pivot(index='Date', columns='reservoirName', values='storagePercentage')
df_res_rain = df_rd.pivot(index='Date', columns='reservoirName', values='rainfallmm')
df_res_level = df_rd.pivot(index='Date', columns='reservoirName', values='waterLevel')

df_res_summary = pd.DataFrame(index=df_res_storage.index)
df_res_summary['National_Avg_Storage_Pct'] = df_res_storage.mean(axis=1)
df_res_summary['National_Total_Rainfall_mm'] = df_res_rain.sum(axis=1)
print(f" -> Reservoir Data loaded: {len(df_res_storage)} days across 12 reservoirs (100% complete).")
print("[SUCCESS] Pipeline consolidation complete.")


---
## Section 0: Comprehensive Summary Tables `[GENERAL]` & `[PROJECT-BIASED]`

The following tables establish the empirical baseline for power dispatch (MW), energy generation (GWh), peak demand regimes, and reservoir hydrology across the 181-day study window.


In [ ]:
# =============================================================================
# SECTION 0: SUMMARY TABLES
# =============================================================================
from IPython.display import display

# -----------------------------------------------------------------------------
# Table 0.1: Power (MW) by Generation Source Summary [GENERAL]
# -----------------------------------------------------------------------------
print("TABLE 0.1: Power (MW) by Generation Source Summary Statistics [GENERAL]")
power_summary = df_lc[SOURCES_MW + ['Total_MW', 'Renewable_MW', 'Fossil_MW', 'Net_Load_MW']].describe().T
power_summary['daily_energy_equiv_gwh'] = (power_summary['mean'] * 24.0) / 1000.0
power_summary = power_summary[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'daily_energy_equiv_gwh']]
power_summary.columns = ['Mean (MW)', 'Std Dev (MW)', 'Min (MW)', 'Q1 (MW)', 'Median (MW)', 'Q3 (MW)', 'Max (MW)', 'Equiv Energy (GWh/day)']
display(power_summary.round(2))

# -----------------------------------------------------------------------------
# Table 0.2: Daily Generation Energy (GWh) by Station Complex Summary [GENERAL]
# -----------------------------------------------------------------------------
print("\nTABLE 0.2: Daily Generation Energy (GWh) by Station Complex [GENERAL]")
energy_cols = [c for c in df_ed_pivot.columns if c not in ['Renewable_GWh', 'Fossil_GWh', 'Total_GWh', 'Renewable_Energy_Share_Pct', 'Fossil_Energy_Share_Pct']]
energy_summary = df_ed_pivot[energy_cols].describe().T
energy_summary['Total Period Energy (GWh)'] = df_ed_pivot[energy_cols].sum()
total_national_gwh = df_ed_pivot['Total_GWh'].sum()
energy_summary['Fuel Mix Share (%)'] = (energy_summary['Total Period Energy (GWh)'] / total_national_gwh) * 100.0
energy_summary = energy_summary[['mean', 'std', 'min', 'max', 'Total Period Energy (GWh)', 'Fuel Mix Share (%)']]
energy_summary.columns = ['Mean (GWh/day)', 'Std Dev (GWh)', 'Min (GWh)', 'Max (GWh)', 'Total GWh (181 Days)', 'Mix Share (%)']
display(energy_summary.round(2))

# -----------------------------------------------------------------------------
# Table 0.3: Peak Demand Regimes Summary [GENERAL]
# -----------------------------------------------------------------------------
print("\nTABLE 0.3: Daily Peak Regimes Operational Statistics [GENERAL]")
peak_summary = pd.DataFrame({
    'Mean Active Power (MW)': df_peak_mw.mean(),
    'Median Active Power (MW)': df_peak_mw.median(),
    'Min Active Power (MW)': df_peak_mw.min(),
    'Max Active Power (MW)': df_peak_mw.max(),
    'Mean Reactive Power (Mvar)': df_peak_mvar.mean(),
    'Max Reactive Power (Mvar)': df_peak_mvar.max()
})
peak_summary['Mean Apparent Power (MVA)'] = np.sqrt(peak_summary['Mean Active Power (MW)']**2 + peak_summary['Mean Reactive Power (Mvar)']**2)
peak_summary['Mean Power Factor'] = peak_summary['Mean Active Power (MW)'] / peak_summary['Mean Apparent Power (MVA)']
display(peak_summary.round(2))

# -----------------------------------------------------------------------------
# Table 0.4: Reservoir Hydrological Metrics Summary [GENERAL]
# -----------------------------------------------------------------------------
print("\nTABLE 0.4: Reservoir Hydrological & Storage Metrics Summary [GENERAL]")
res_summary_tbl = pd.DataFrame({
    'Mean Storage (%)': df_res_storage.mean(),
    'Min Storage (%)': df_res_storage.min(),
    'Max Storage (%)': df_res_storage.max(),
    'Storage Range (%)': df_res_storage.max() - df_res_storage.min(),
    'Total Rainfall (mm)': df_res_rain.sum(),
    'Max 24h Rain (mm)': df_res_rain.max(),
    'Mean Water Level': df_res_level.mean()
})
display(res_summary_tbl.round(2))

# -----------------------------------------------------------------------------
# Table 0.5: Renewable vs. Fossil Fuel System Balance [PROJECT-BIASED]
# -----------------------------------------------------------------------------
print("\nTABLE 0.5: Renewable vs. Fossil Fuel Generation Balance [PROJECT-BIASED]")
balance_tbl = pd.DataFrame({
    'Metric': [
        'Total Generation Energy (GWh)',
        'Daily Mean Generation (GWh/day)',
        'Minimum Daily Generation (GWh)',
        'Maximum Daily Generation (GWh)',
        'Mean Instantaneous Power (MW)',
        'Minimum Instantaneous Power (MW)',
        'Maximum Instantaneous Power (MW)',
        'Overall Generation Share (%)'
    ],
    'Renewable': [
        df_ed_full['Renewable_GWh'].sum(),
        df_ed_full['Renewable_GWh'].mean(),
        df_ed_full['Renewable_GWh'].min(),
        df_ed_full['Renewable_GWh'].max(),
        df_lc['Renewable_MW'].mean(),
        df_lc['Renewable_MW'].min(),
        df_lc['Renewable_MW'].max(),
        (df_ed_full['Renewable_GWh'].sum() / df_ed_full['Total_GWh'].sum()) * 100.0
    ],
    'Fossil Fuel': [
        df_ed_full['Fossil_GWh'].sum(),
        df_ed_full['Fossil_GWh'].mean(),
        df_ed_full['Fossil_GWh'].min(),
        df_ed_full['Fossil_GWh'].max(),
        df_lc['Fossil_MW'].mean(),
        df_lc['Fossil_MW'].min(),
        df_lc['Fossil_MW'].max(),
        (df_ed_full['Fossil_GWh'].sum() / df_ed_full['Total_GWh'].sum()) * 100.0
    ]
})
display(balance_tbl.round(2))


---
## Section 1: Distribution Analysis & Histograms `[GENERAL]` & `[PROJECT-BIASED]`

This section evaluates the probabilistic distributions of daily national energy, renewable penetration, peak solar capacity, night-peak fossil dependence, and individual generator outputs.


In [ ]:
# =============================================================================
# SECTION 1: HISTOGRAMS & DISTRIBUTION PLOTS
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Key System Distribution Indicators [GENERAL] & [PROJECT-BIASED]", fontsize=16, fontweight='bold', y=0.98)

# 1.1 Daily Total Generation (GWh) [GENERAL]
ax = axes[0, 0]
sns.histplot(df_ed_full['Total_GWh'], kde=True, bins=25, color=COLOR_TOTAL, ax=ax, edgecolor='black', alpha=0.6)
mean_gen = df_ed_full['Total_GWh'].mean()
median_gen = df_ed_full['Total_GWh'].median()
ax.axvline(mean_gen, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_gen:.1f} GWh')
ax.axvline(median_gen, color='black', linestyle=':', linewidth=1.5, label=f'Median: {median_gen:.1f} GWh')
ax.set_title("1.1 Daily Total Generation Distribution [GENERAL]", fontweight='bold')
ax.set_xlabel("Daily Total Generation (GWh)")
ax.set_ylabel("Frequency (Days)")
ax.legend()

# 1.2 Daily Renewable Share (%) [PROJECT-BIASED]
ax = axes[0, 1]
sns.histplot(df_ed_full['Renewable_Energy_Share_Pct'], kde=True, bins=25, color=COLOR_RENEWABLE, ax=ax, edgecolor='black', alpha=0.6)
mean_ren = df_ed_full['Renewable_Energy_Share_Pct'].mean()
ax.axvline(mean_ren, color='darkgreen', linestyle='--', linewidth=1.5, label=f'Mean Share: {mean_ren:.1f}%')
ax.axvline(50.0, color='gray', linestyle=':', linewidth=1.5, label='50% Threshold')
ax.set_title("1.2 Daily Renewable Generation Share Distribution [PROJECT-BIASED]", fontweight='bold')
ax.set_xlabel("Renewable Generation Share (%)")
ax.set_ylabel("Frequency (Days)")
ax.legend()

# 1.3 Daily Peak Solar Power (MW) [PROJECT-BIASED]
daily_peak_solar = df_lc.groupby('Date')['Solar'].max()
ax = axes[1, 0]
sns.histplot(daily_peak_solar, kde=True, bins=25, color=COLOR_SOLAR, ax=ax, edgecolor='black', alpha=0.6)
mean_solar = daily_peak_solar.mean()
ax.axvline(mean_solar, color='darkorange', linestyle='--', linewidth=1.5, label=f'Mean Peak: {mean_solar:.1f} MW')
ax.set_title("1.3 Daily Peak Solar Capacity Distribution [PROJECT-BIASED]", fontweight='bold')
ax.set_xlabel("Instantaneous Peak Solar Power (MW)")
ax.set_ylabel("Frequency (Days)")
ax.legend()

# 1.4 Night-Peak Fossil Fuel Share (%) [PROJECT-BIASED]
ax = axes[1, 1]
sns.histplot(df_np_mw['Fossil_NP_Share_Pct'], kde=True, bins=25, color=COLOR_FOSSIL, ax=ax, edgecolor='black', alpha=0.6)
mean_np_fossil = df_np_mw['Fossil_NP_Share_Pct'].mean()
ax.axvline(mean_np_fossil, color='darkred', linestyle='--', linewidth=1.5, label=f'Mean Fossil Share: {mean_np_fossil:.1f}%')
ax.set_title("1.4 Night-Peak Fossil Fuel Share Distribution [PROJECT-BIASED]", fontweight='bold')
ax.set_xlabel("Fossil Fuel Power Share at 19:00 Night Peak (%)")
ax.set_ylabel("Frequency (Days)")
ax.legend()

plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------------
# 1.5 Individual Generation Distributions (5 Subplots) [GENERAL]
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
fig.suptitle("1.5 Individual Generation Distributions by Source (GWh/Day) [GENERAL]", fontsize=15, fontweight='bold', y=1.02)

# Estimate daily energy by source from Load Curve (15-min intervals: sum(MW * 0.25h) / 1000)
df_lc_daily_energy = df_lc.groupby('Date')[SOURCES_MW].apply(lambda g: (g * 0.25).sum() / 1000.0)

source_subset = [('Solar', COLOR_SOLAR), ('Wind', COLOR_WIND), ('Major Hydro', COLOR_HYDRO), 
                 ('Coal', COLOR_COAL), ('Thermal-Oil', COLOR_THERMAL)]

for i, (src, color) in enumerate(source_subset):
    ax = axes[i]
    sns.histplot(df_lc_daily_energy[src], kde=True, bins=20, color=color, ax=ax, edgecolor='black', alpha=0.6)
    m = df_lc_daily_energy[src].mean()
    ax.axvline(m, color='black', linestyle='--', linewidth=1.5, label=f'Mean: {m:.1f}')
    ax.set_title(f"{src}", fontweight='bold')
    ax.set_xlabel("Daily Generation (GWh)")
    ax.set_ylabel("Days" if i == 0 else "")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------------
# 1.6 Renewable vs. Non-Renewable Generation Distributions [PROJECT-BIASED]
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle("1.6 Renewable vs. Non-Renewable Generation Distributions [PROJECT-BIASED]", fontsize=15, fontweight='bold')

# Renewable
ax = axes[0]
sns.histplot(df_ed_full['Renewable_GWh'], kde=True, bins=25, color=COLOR_RENEWABLE, ax=ax, edgecolor='black', alpha=0.6)
ax.axvline(df_ed_full['Renewable_GWh'].mean(), color='darkgreen', linestyle='--', label=f"Mean: {df_ed_full['Renewable_GWh'].mean():.1f} GWh")
ax.set_title("Renewable Generation Distribution", fontweight='bold')
ax.set_xlabel("Daily Renewable Energy (GWh)")
ax.set_ylabel("Frequency (Days)")
ax.legend()

# Non-Renewable (Fossil)
ax = axes[1]
sns.histplot(df_ed_full['Fossil_GWh'], kde=True, bins=25, color=COLOR_FOSSIL, ax=ax, edgecolor='black', alpha=0.6)
ax.axvline(df_ed_full['Fossil_GWh'].mean(), color='darkred', linestyle='--', label=f"Mean: {df_ed_full['Fossil_GWh'].mean():.1f} GWh")
ax.set_title("Non-Renewable (Fossil) Generation Distribution", fontweight='bold')
ax.set_xlabel("Daily Fossil Energy (GWh)")
ax.set_ylabel("Frequency (Days)")
ax.legend()

plt.tight_layout()
plt.show()


---
## Section 2: Comparative Distributions & Notched Boxplots `[GENERAL]` & `[PROJECT-BIASED]`

Notched boxplots allow rigorous comparison of medians and variances. If the notches between two boxes do not overlap, there is strong statistical evidence (95% confidence) that their true medians differ.


In [ ]:
# =============================================================================
# SECTION 2: NOTCHED BOXPLOTS
# =============================================================================
# 2.1 Power Dispatch by Source (MW) [GENERAL]
plt.figure(figsize=(14, 6))
df_lc_melt = df_lc.melt(value_vars=SOURCES_MW, var_name='Source', value_name='Power_MW')
palette_list = [SOURCE_PALETTE[s] for s in SOURCES_MW]

ax = sns.boxplot(data=df_lc_melt, x='Source', y='Power_MW', notch=True, palette=palette_list, 
                 showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
plt.title("2.1 Power Dispatch by Generation Source (MW) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Generation Source")
plt.ylabel("Active Power Dispatch (MW)")
plt.xticks(rotation=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2.2 Peak Power by Source at Night Peak (MW) [GENERAL]
plt.figure(figsize=(15, 6))
np_sources = [c for c in df_npd['stationGroupName'].unique() if c != 'Capacitors']
ax = sns.boxplot(data=df_npd[df_npd['stationGroupName'].isin(np_sources)], 
                 x='stationGroupName', y='activePowerMW', notch=True, palette='tab10',
                 showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
plt.title("2.2 Peak Power by Station Group at Night Peak (19:00 MW) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Station Group / Complex")
plt.ylabel("Active Power at Night Peak (MW)")
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2.3 Daily Energy Generation by Station Complex (GWh) [GENERAL]
plt.figure(figsize=(15, 6))
ax = sns.boxplot(data=df_ed, x='stationGroupName', y='energyGWh', notch=True, palette='Set2',
                 showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
plt.title("2.3 Daily Energy Generation by Station Complex (GWh) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Station Complex")
plt.ylabel("Daily Energy Generation (GWh)")
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 2.4 Fossil Fuel Power Share of Night Peak (%) [PROJECT-BIASED]
# 2.5 Day, Night, and Off-Peak Active Demand (MW) [GENERAL]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 2.4
ax = axes[0]
sns.boxplot(y=df_np_mw['Fossil_NP_Share_Pct'], notch=True, color=COLOR_FOSSIL, ax=ax, width=0.4,
            showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
sns.stripplot(y=df_np_mw['Fossil_NP_Share_Pct'], color='black', alpha=0.3, jitter=0.15, size=4, ax=ax)
ax.set_title("2.4 Fossil Fuel Power Share of Night Peak [PROJECT-BIASED]", fontweight='bold')
ax.set_ylabel("Fossil Fuel Power Share (%)")
ax.grid(True, linestyle='--', alpha=0.5)

# Plot 2.5
ax = axes[1]
df_peak_melt = df_peak_mw[['Off Peak', 'Day Peak', 'Night Peak']].melt(var_name='Regime', value_name='Active_MW')
sns.boxplot(data=df_peak_melt, x='Regime', y='Active_MW', notch=True, palette=REGIME_PALETTE, ax=ax,
            showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
ax.set_title("2.5 Day, Night, and Off-Peak Active Power Comparison [GENERAL]", fontweight='bold')
ax.set_xlabel("Operational Peak Regime")
ax.set_ylabel("Active Power (MW)")
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


---
## Section 3: Kernel Density & Violin Plots `[GENERAL]`

Violin plots combine boxplots with rotated kernel density estimates, revealing bimodal distributions, skewness, and multi-peak dispatch behaviors across generating stations and reservoirs.


In [ ]:
# =============================================================================
# SECTION 3: VIOLIN PLOTS
# =============================================================================
# 3.1 Daily Energy Generation Distributions by Complex [GENERAL]
plt.figure(figsize=(16, 6))
sns.violinplot(data=df_ed, x='stationGroupName', y='energyGWh', palette='tab10', cut=0, inner='quartile')
plt.title("3.1 Daily Generation Distributions by Station Complex (GWh) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Station Complex")
plt.ylabel("Daily Energy Generation (GWh)")
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 3.2 Storage Percentage by Reservoir [GENERAL]
plt.figure(figsize=(16, 6))
sns.violinplot(data=df_rd, x='reservoirName', y='storagePercentage', palette='Blues_r', cut=0, inner='quartile')
plt.title("3.2 Usable Storage Percentage Distribution by Reservoir (%) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Hydro Catchment Reservoir")
plt.ylabel("Storage Percentage (%)")
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


---
## Section 4: 24-Hour Diurnal Dynamics (15-Minute Resolution) `[GENERAL]` & `[PROJECT-BIASED]`

Analyzes the diurnal cycle of the Sri Lankan power system, highlighting the base load, solar curve ramp, evening peak timing, and the potential window for solar-excess back pumping.


In [ ]:
# =============================================================================
# SECTION 4: 24-HOUR DIURNAL TRENDS (15-MIN RESOLUTION)
# =============================================================================
# Prepare diurnal time groupings
diurnal_mean = df_lc.groupby('Time_Str')[SOURCES_MW + ['Total_MW', 'Renewable_MW', 'Fossil_MW', 'Net_Load_MW']].mean()
time_labels = diurnal_mean.index.tolist()
x_indices = np.arange(len(time_labels))

# 4.1 Load Curve Stacked Area Chart [GENERAL]
plt.figure(figsize=(16, 6))
stack_y = [diurnal_mean[s].to_numpy() for s in SOURCES_MW]
plt.stackplot(x_indices, *stack_y, labels=SOURCES_MW, colors=[SOURCE_PALETTE[s] for s in SOURCES_MW], alpha=0.9)
plt.title("4.1 Average 24-Hour National Load Curve Dispatch Stack [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (15-Minute Intervals)")
plt.ylabel("Active Power Dispatch (MW)")
tick_steps = 8 # every 2 hours
plt.xticks(x_indices[::tick_steps], [t[:5] for t in time_labels[::tick_steps]])
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=True)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 4.2 Total Power Demand with 95% Confidence Interval Line Chart [GENERAL]
plt.figure(figsize=(16, 6))
sns.lineplot(data=df_lc, x='Time_Str', y='Total_MW', color=COLOR_TOTAL, linewidth=2.0, 
             errorbar=('ci', 95), label='Total Grid Demand (Mean ± 95% CI)')

# Highlight Day Peak Window (15:30 to 17:00)
day_start, day_end = '15:30:00', '17:00:00'
plt.axvspan(day_start, day_end, color='#F59E0B', alpha=0.22, label='Primary Day Peak Window (15:30 - 17:00)')

# Highlight Night Peak Window (18:30 to 20:00)
night_start, night_end = '18:30:00', '20:00:00'
plt.axvspan(night_start, night_end, color='#DC2626', alpha=0.22, label='Primary Night Peak Window (18:30 - 20:00)')

plt.title("4.2 Total Power Demand Profile with 95% Confidence Interval & Peak Windows [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (HH:MM)")
plt.ylabel("Total Active Power (MW)")
plt.xticks(time_labels[::tick_steps], [t[:5] for t in time_labels[::tick_steps]])
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 4.3 Power Generation by Source Multiple Line Chart [GENERAL]
plt.figure(figsize=(16, 6))
for s in SOURCES_MW:
    plt.plot(x_indices, diurnal_mean[s].to_numpy(), label=s, color=SOURCE_PALETTE[s], linewidth=2.0)
plt.title("4.3 Power Generation by Source Diurnal Dispatch (MW) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day")
plt.ylabel("Mean Generation (MW)")
plt.xticks(x_indices[::tick_steps], [t[:5] for t in time_labels[::tick_steps]])
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 4.4, 4.5, 4.6 Diurnal Focus Charts [PROJECT-BIASED]
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# 4.4 Solar Curve Line Chart
ax = axes[0]
solar_stats = df_lc.groupby('Time_Str')['Solar'].agg(['mean', 'std'])
mean_arr = solar_stats['mean'].to_numpy()
std_arr = solar_stats['std'].to_numpy()
ax.plot(x_indices, mean_arr, color=COLOR_SOLAR, linewidth=2.5, label='Solar Power (Mean)')
ax.fill_between(x_indices, np.maximum(0, mean_arr - std_arr), mean_arr + std_arr, color=COLOR_SOLAR, alpha=0.25, label='±1 Standard Deviation')
ax.set_title("4.4 Diurnal Solar Generation Curve [PROJECT-BIASED]", fontweight='bold')
ax.set_ylabel("Solar Power (MW)")
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.4)

# 4.5 Solar, Non-Solar, Total Multiple Line Chart
ax = axes[1]
ax.plot(x_indices, diurnal_mean['Total_MW'].to_numpy(), color=COLOR_TOTAL, linewidth=2.0, linestyle='--', label='Total Demand')
ax.plot(x_indices, (diurnal_mean['Total_MW'] - diurnal_mean['Solar']).to_numpy(), color='#475569', linewidth=2.0, label='Non-Solar Demand (Net of Solar)')
ax.plot(x_indices, diurnal_mean['Solar'].to_numpy(), color=COLOR_SOLAR, linewidth=2.0, label='Solar Generation')
ax.set_title("4.5 Solar Influx vs. Total Demand & Non-Solar Requirement [PROJECT-BIASED]", fontweight='bold')
ax.set_ylabel("Active Power (MW)")
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.4)

# 4.6 Renewables, Fossil Fuel Multiple Line Chart
ax = axes[2]
ax.plot(x_indices, diurnal_mean['Renewable_MW'].to_numpy(), color=COLOR_RENEWABLE, linewidth=2.2, label='Renewable Generation (MW)')
ax.plot(x_indices, diurnal_mean['Fossil_MW'].to_numpy(), color=COLOR_FOSSIL, linewidth=2.2, label='Fossil Fuel Generation (MW)')
ax.plot(x_indices, diurnal_mean['Total_MW'].to_numpy(), color=COLOR_TOTAL, linestyle=':', linewidth=1.8, label='Total Demand (MW)')
ax.set_title("4.6 Diurnal Renewables vs. Fossil Fuel Balance [PROJECT-BIASED]", fontweight='bold')
ax.set_xlabel("Time of Day")
ax.set_ylabel("Active Power (MW)")
ax.set_xticks(x_indices[::tick_steps])
ax.set_xticklabels([t[:5] for t in time_labels[::tick_steps]])
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


---
## Section 5: Date Series Trends & Longitudinal Seasonality `[GENERAL]` & `[PROJECT-BIASED]`

Tracks system evolution across all 181 calendar days, revealing seasonal hydrology, monsoon transitions, fossil reduction patterns, and weekend demand drops.


In [ ]:
# =============================================================================
# SECTION 5: DATE SERIES TRENDS & LONGITUDINAL SEASONALITY
# =============================================================================
# 5.1 Date (rows) x Slot (columns) Total Power MW Heatmap [GENERAL]
plt.figure(figsize=(16, 8))
heatmap_data = df_lc.pivot(index='Date', columns='Time_Str', values='Total_MW')
ax = sns.heatmap(heatmap_data, cmap='viridis', cbar_kws={'label': 'Total Grid Demand (MW)'})
plt.title("5.1 2D Date (Rows) x Time Slot (Columns) Load Intensity Heatmap (MW) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (15-Minute Intervals)")
plt.ylabel("Date (Jan 27 - Jul 26, 2026)")
x_ticks = np.arange(0, len(heatmap_data.columns), 8)
plt.xticks(x_ticks, [heatmap_data.columns[i][:5] for i in x_ticks], rotation=0)
y_ticks = np.arange(0, len(heatmap_data.index), 15)
plt.yticks(y_ticks, [str(heatmap_data.index[i]) for i in y_ticks], rotation=0)
plt.tight_layout()
plt.show()

# 5.2 Daily Peaks Multiple Line Chart [GENERAL]
plt.figure(figsize=(16, 5))
plt.plot(df_peak_mw.index, df_peak_mw['Day Peak'], color=REGIME_PALETTE['Day Peak'], label='Day Peak Active Power (MW)', linewidth=1.8)
plt.plot(df_peak_mw.index, df_peak_mw['Night Peak'], color=REGIME_PALETTE['Night Peak'], label='Night Peak Active Power (MW)', linewidth=2.0)
plt.plot(df_peak_mw.index, df_peak_mw['Off Peak'], color=REGIME_PALETTE['Off Peak'], label='Off Peak Active Power (MW)', linewidth=1.8)
plt.title("5.2 Daily Peak Regimes Trajectory (MW) vs. Date [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Active Power (MW)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 5.3 Daily Peak Occurrence Timing vs. Date [GENERAL]
df_peak_times_clean = df_pd_clean.pivot(index='Date', columns='stationGroupName', values='Time')
df_peak_times_clean = df_peak_times_clean.reindex(all_dates).ffill()

plt.figure(figsize=(16, 5))
day_peak_hours = df_peak_times_clean['Day Peak'].apply(lambda t: t.hour + t.minute/60.0 if pd.notna(t) else np.nan)
night_peak_hours = df_peak_times_clean['Night Peak'].apply(lambda t: t.hour + t.minute/60.0 if pd.notna(t) else np.nan)

plt.scatter(df_peak_times_clean.index, day_peak_hours, color=REGIME_PALETTE['Day Peak'], label='Day Peak Time of Day', alpha=0.7, s=25)
plt.scatter(df_peak_times_clean.index, night_peak_hours, color=REGIME_PALETTE['Night Peak'], label='Night Peak Time of Day', alpha=0.7, s=25)
plt.yticks(np.arange(11, 22), [f"{h:02d}:00" for h in range(11, 22)])
plt.title("5.3 Daily Peak Occurrence Clock Timings vs. Date [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Time of Occurrence")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 5.4 Total Energy vs. Date [GENERAL] & 5.5 Reservoir Storage % vs. Date [GENERAL]
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# 5.4 Total Energy vs Date
ax = axes[0]
ax.plot(df_ed_full.index, df_ed_full['Total_GWh'], color=COLOR_TOTAL, label='Daily Total Generation (GWh)', linewidth=1.5)
ax.plot(df_ed_full.index, df_ed_full['Total_GWh'].rolling(7, min_periods=1).mean(), color='red', label='7-Day Rolling Mean', linewidth=2.0)
ax.set_title("5.4 Total Daily Generation Energy (GWh) vs. Date [GENERAL]", fontweight='bold')
ax.set_ylabel("Total Generation (GWh)")
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)

# 5.5 Reservoir Percentages vs Date
ax = axes[1]
for res in df_res_storage.columns:
    ax.plot(df_res_storage.index, df_res_storage[res], label=res, alpha=0.7, linewidth=1.2)
ax.plot(df_res_summary.index, df_res_summary['National_Avg_Storage_Pct'], color='black', linewidth=2.5, linestyle='--', label='National Average')
ax.set_title("5.5 Catchment Reservoir Usable Storage Percentages (%) vs. Date [GENERAL]", fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Storage Percentage (%)")
ax.legend(bbox_to_anchor=(1.01, 1.0), loc='upper left', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

# 5.6, 5.7, 5.8 Peak Power by Specific Sources vs. Date [PROJECT-BIASED]
daily_peak_by_source = df_lc.groupby('Date')[SOURCES_MW].max()

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

axes[0].plot(daily_peak_by_source.index, daily_peak_by_source['Solar'], color=COLOR_SOLAR, linewidth=1.8)
axes[0].set_title("5.6 Daily Solar Peak Power (MW) vs. Date [PROJECT-BIASED]", fontweight='bold')
axes[0].set_ylabel("Solar Peak (MW)")
axes[0].grid(True, linestyle='--', alpha=0.4)

axes[1].plot(daily_peak_by_source.index, daily_peak_by_source['Major Hydro'], color=COLOR_HYDRO, linewidth=1.8)
axes[1].set_title("5.7 Daily Major Hydro Peak Power (MW) vs. Date [PROJECT-BIASED]", fontweight='bold')
axes[1].set_ylabel("Major Hydro Peak (MW)")
axes[1].grid(True, linestyle='--', alpha=0.4)

axes[2].plot(daily_peak_by_source.index, daily_peak_by_source['Thermal-Oil'], color=COLOR_THERMAL, linewidth=1.8)
axes[2].set_title("5.8 Daily Thermal-Oil Peak Power (MW) vs. Date [PROJECT-BIASED]", fontweight='bold')
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Thermal-Oil Peak (MW)")
axes[2].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

# 5.9 Renewables vs. Fossil Fuel Generation (GWh) vs. Date [PROJECT-BIASED]
plt.figure(figsize=(16, 5.5))
plt.plot(df_ed_full.index, df_ed_full['Total_GWh'], color=COLOR_TOTAL, linestyle='--', linewidth=2.0, label='Total')
plt.plot(df_ed_full.index, df_ed_full['Fossil_GWh'], color=COLOR_FOSSIL, linewidth=2.0, label='Fossil Fuel')
plt.plot(df_ed_full.index, df_ed_full['Renewable_GWh'], color=COLOR_RENEWABLE, linewidth=2.0, label='Renewable')
plt.title("5.9 Renewables vs. Fossil Fuel Generation (GWh) vs. Date [PROJECT-BIASED]", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Energy (GWh)")
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=True)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# 5.10 Weekday vs. Weekend Mean Profile with Bands [PROJECT-BIASED]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# Total Load Profile
ax = axes[0]
sns.lineplot(data=df_lc, x='Time_Str', y='Total_MW', hue='IsWeekend', 
             palette={False: '#1E293B', True: '#DC2626'}, ax=ax, errorbar=('ci', 95), linewidth=2.0)
ax.set_title("Total Load: Weekday vs. Weekend [PROJECT-BIASED]", fontweight='bold')
ax.set_xlabel("Time of Day")
ax.set_ylabel("Total Demand (MW)")
ax.set_xticks(time_labels[::tick_steps])
ax.set_xticklabels([t[:5] for t in time_labels[::tick_steps]])
ax.legend(['Weekday (Mon-Fri)', 'Weekend (Sat-Sun)'])
ax.grid(True, linestyle='--', alpha=0.4)

# Net Load Profile (Total - Solar - Wind)
ax = axes[1]
sns.lineplot(data=df_lc, x='Time_Str', y='Net_Load_MW', hue='IsWeekend', 
             palette={False: '#1E293B', True: '#DC2626'}, ax=ax, errorbar=('ci', 95), linewidth=2.0)
ax.set_title("Net Load (Total - Solar - Wind): Weekday vs. Weekend [PROJECT-BIASED]", fontweight='bold')
ax.set_xlabel("Time of Day")
ax.set_xticks(time_labels[::tick_steps])
ax.set_xticklabels([t[:5] for t in time_labels[::tick_steps]])
ax.legend(['Weekday (Mon-Fri)', 'Weekend (Sat-Sun)'])
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

# 5.11 Solar Share % vs Date [PROJECT-BIASED]
plt.figure(figsize=(16, 4.5))
solar_share_daily = (df_lc.groupby('Date')['Solar'].sum() / df_lc.groupby('Date')['Total_MW'].sum()) * 100.0
plt.plot(solar_share_daily.index, solar_share_daily, color=COLOR_SOLAR, linewidth=1.8, label='Solar Share (%)')
plt.plot(solar_share_daily.index, solar_share_daily.rolling(7, min_periods=1).mean(), color='black', linewidth=2.0, linestyle='--', label='7-Day Rolling Mean')
plt.title("5.11 Solar Share Percentage of Total Daily Generation vs. Date [PROJECT-BIASED]", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Solar Share of Daily Energy (%)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 5.12 Renewable vs. Fossil Fuel Generation Share Percentage Stacked Bar Chart [PROJECT-BIASED]
plt.figure(figsize=(18, 6.5))
dates_idx = np.arange(len(df_ed_full))
ren_pct = df_ed_full['Renewable_Energy_Share_Pct'].to_numpy()
fos_pct = df_ed_full['Fossil_Energy_Share_Pct'].to_numpy()

bar_width = 0.65
plt.bar(dates_idx, ren_pct, width=bar_width, color=COLOR_RENEWABLE, label='Renewable')
plt.bar(dates_idx, fos_pct, bottom=ren_pct, width=bar_width, color=COLOR_FOSSIL, label='Fossil Fuel')

for idx in range(0, len(dates_idx), 7):
    r_val = float(ren_pct[idx])
    f_val = float(fos_pct[idx])
    plt.text(idx, r_val / 2.0, f"{r_val:.0f}%", ha='center', va='center', color='white', fontsize=7.5, fontweight='bold')
    plt.text(idx, r_val + (f_val / 2.0), f"{f_val:.0f}%", ha='center', va='center', color='white', fontsize=7.5, fontweight='bold')

plt.title("5.12 Daily Renewable vs. Fossil Fuel Generation Share Percentage (%) [PROJECT-BIASED]", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Energy (%)")
plt.ylim(0, 105)
plt.yticks([0, 25, 50, 75, 100], ['0%', '25%', '50%', '75%', '100%'])
date_ticks = np.arange(0, len(dates_idx), 14)
plt.xticks(date_ticks, [str(df_ed_full.index[i]) for i in date_ticks], rotation=25)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=True)
plt.grid(axis='y', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()


---
## Section 6: Day-of-Week Operational Patterns `[GENERAL]`

Evaluates weekly demand cyclicality across Monday through Sunday. Uncovers whether weekends present low-demand surplus periods suitable for base-load charging or pumped hydro energy storage.


In [ ]:
# =============================================================================
# SECTION 6: DAY-OF-WEEK OPERATIONAL PATTERNS [GENERAL]
# =============================================================================
df_ed_full['DayOfWeek'] = pd.to_datetime(df_ed_full.index).day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# 6.1 Total Generation (GWh) by Day of Week [GENERAL]
ax = axes[0]
sns.boxplot(data=df_ed_full, x='DayOfWeek', y='Total_GWh', order=day_order, palette='Blues', ax=ax,
            showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
ax.set_title("6.1 Total Daily Generation (GWh) by Day of Week [GENERAL]", fontweight='bold')
ax.set_xlabel("Day of the Week")
ax.set_ylabel("Total Generation (GWh)")
ax.set_xticklabels(day_order, rotation=30)
ax.grid(True, linestyle='--', alpha=0.4)

# 6.2 Peak Power Data by Day of Week [GENERAL]
ax = axes[1]
df_peak_dow = df_peak_mw.copy()
df_peak_dow['DayOfWeek'] = pd.to_datetime(df_peak_dow.index).day_name()
df_peak_dow_melt = df_peak_dow.melt(id_vars='DayOfWeek', value_vars=['Day Peak', 'Night Peak'], 
                                   var_name='Peak_Regime', value_name='Power_MW')

sns.barplot(data=df_peak_dow_melt, x='DayOfWeek', y='Power_MW', hue='Peak_Regime', 
            order=day_order, palette=[REGIME_PALETTE['Day Peak'], REGIME_PALETTE['Night Peak']], ax=ax, ci=95)
ax.set_title("6.2 Peak Power Demand by Day of Week (MW) [GENERAL]", fontweight='bold')
ax.set_xlabel("Day of the Week")
ax.set_ylabel("Mean Peak Power (MW)")
ax.set_xticklabels(day_order, rotation=30)
ax.set_ylim(2000, 3100)
ax.legend(title="Regime")
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


---
## Section 7: Monthly & Category Breakdowns `[GENERAL]` & `[PROJECT-BIASED]`

Provides monthly aggregated 100% fuel mixes, reservoir spatial-temporal heatmaps, and day/night peak dispatch breakdowns across the study horizon.


In [ ]:
# =============================================================================
# SECTION 7: MONTHLY BREAKDOWNS & PEAK STACK CHARTS
# =============================================================================
# 7.1 Monthly % by Category (100% Stacked Bar) [GENERAL]
df_ed_monthly = df_ed_pivot.copy()
df_ed_monthly['Month'] = pd.to_datetime(df_ed_monthly.index).strftime('%Y-%m')
monthly_energy = df_ed_monthly.groupby('Month')[energy_cols].sum()
monthly_energy_pct = (monthly_energy.T / monthly_energy.sum(axis=1)).T * 100.0

plt.figure(figsize=(14, 6))
bottom = np.zeros(len(monthly_energy_pct))
months = monthly_energy_pct.index.tolist()
cmap = plt.get_cmap('tab20', len(energy_cols))

for i, col in enumerate(energy_cols):
    vals = monthly_energy_pct[col].values
    plt.bar(months, vals, bottom=bottom, label=col, color=cmap(i), width=0.55, edgecolor='black', linewidth=0.5)
    bottom += vals

plt.title("7.1 Monthly Energy Generation Mix (% Share by Category) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Generation Share (%)")
plt.ylim(0, 100)
plt.legend(bbox_to_anchor=(1.01, 1.0), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 7.2 Reservoir (rows) x Date, Storage % Heatmap [GENERAL]
plt.figure(figsize=(18, 6.5))
ax = sns.heatmap(df_res_storage.T, cmap='Blues', cbar_kws={'label': 'Storage Percentage (%)'})
plt.title("7.2 Reservoir Storage Percentage Temporal Heatmap (%) [GENERAL]", fontsize=14, fontweight='bold')
plt.xlabel("Date (Jan 27 - Jul 26, 2026)")
plt.ylabel("Catchment Reservoir")
date_ticks = np.arange(0, len(df_res_storage.index), 14)
plt.xticks(date_ticks, [str(df_res_storage.index[i]) for i in date_ticks], rotation=25)
plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------------
# 7.3 & 7.4 Stacked Area & Stacked Bar Charts of Power at Day & Night Peak [PROJECT-BIASED]
# -----------------------------------------------------------------------------
df_lc_day_peak = df_lc[(df_lc['Time_Str'] >= '15:30:00') & (df_lc['Time_Str'] <= '17:00:00')].sort_values('Total_MW', ascending=False).drop_duplicates('Date').set_index('Date')[SOURCES_MW].reindex(all_dates).interpolate()
df_lc_night_peak = df_lc[(df_lc['Time_Str'] >= '18:30:00') & (df_lc['Time_Str'] <= '20:00:00')].sort_values('Total_MW', ascending=False).drop_duplicates('Date').set_index('Date')[SOURCES_MW].reindex(all_dates).interpolate()

# 7.3 Two Stacked Area Charts (Power by Source at Day Peak and Night Peak)
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
x_p_indices = np.arange(len(df_lc_day_peak))

# Day Peak Stacked Area
axes[0].stackplot(x_p_indices, 
                 *[df_lc_day_peak[s].to_numpy() for s in SOURCES_MW],
                 labels=SOURCES_MW,
                 colors=[SOURCE_PALETTE[s] for s in SOURCES_MW],
                 alpha=0.9)
axes[0].set_title("7.3 (A) Power Dispatch by Source at Day Peak (Stacked Area) [PROJECT-BIASED]", fontweight='bold')
axes[0].set_ylabel("Power (MW)")
axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
axes[0].grid(True, linestyle='--', alpha=0.4)

# Night Peak Stacked Area
axes[1].stackplot(x_p_indices, 
                 *[df_lc_night_peak[s].to_numpy() for s in SOURCES_MW],
                 labels=SOURCES_MW,
                 colors=[SOURCE_PALETTE[s] for s in SOURCES_MW],
                 alpha=0.9)
axes[1].set_title("7.3 (B) Power Dispatch by Source at Night Peak (Stacked Area) [PROJECT-BIASED]", fontweight='bold')
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Power (MW)")
axes[1].set_xticks(x_p_indices[::14])
axes[1].set_xticklabels([str(all_dates[i]) for i in x_p_indices[::14]], rotation=25)
axes[1].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

# 7.4 Two Stacked Bar Charts (Power by Source at Day Peak and Night Peak)
sample_dates = all_dates[::3]
df_bar_day = df_lc_day_peak.loc[sample_dates]
df_bar_night = df_lc_night_peak.loc[sample_dates]

fig, axes = plt.subplots(2, 1, figsize=(18, 11), sharex=True)
bar_x = np.arange(len(sample_dates))

# Day Peak Stacked Bar
bottom = np.zeros(len(sample_dates))
for s in SOURCES_MW:
    axes[0].bar(bar_x, df_bar_day[s].to_numpy(), bottom=bottom, label=s, color=SOURCE_PALETTE[s], width=0.7, edgecolor='black', linewidth=0.3)
    bottom += df_bar_day[s].to_numpy()
axes[0].set_title("7.4 (A) Power Dispatch by Source at Day Peak (Stacked Bar, 3-Day Sample) [PROJECT-BIASED]", fontweight='bold')
axes[0].set_ylabel("Active Power (MW)")
axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Night Peak Stacked Bar
bottom = np.zeros(len(sample_dates))
for s in SOURCES_MW:
    axes[1].bar(bar_x, df_bar_night[s].to_numpy(), bottom=bottom, label=s, color=SOURCE_PALETTE[s], width=0.7, edgecolor='black', linewidth=0.3)
    bottom += df_bar_night[s].to_numpy()
axes[1].set_title("7.4 (B) Power Dispatch by Source at Night Peak (Stacked Bar, 3-Day Sample) [PROJECT-BIASED]", fontweight='bold')
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Active Power (MW)")
axes[1].set_xticks(bar_x[::3])
axes[1].set_xticklabels([str(sample_dates[i]) for i in range(0, len(sample_dates), 3)], rotation=25)
axes[1].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


---
## Section 8: Linear & Non-Linear Correlation Analysis `[GENERAL]` & `[PROJECT-BIASED]`

Compares Pearson linear correlation (parametric linear dependencies) and Spearman rank correlation (monotonic non-linear relationships) across generation sources, peak hours, and catchment hydrology.


In [ ]:
# =============================================================================
# SECTION 8: CORRELATION HEATMAPS (LINEAR PEARSON & NON-LINEAR SPEARMAN)
# =============================================================================
# 8.1 Power by Source General Correlation Heatmaps [GENERAL]
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
corr_p = df_lc[SOURCES_MW].corr(method='pearson')
corr_s = df_lc[SOURCES_MW].corr(method='spearman')

sns.heatmap(corr_p, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0], cbar_kws={'label': 'Pearson r'})
axes[0].set_title("8.1 (A) Power by Source: Pearson Linear Correlation [GENERAL]", fontweight='bold')

sns.heatmap(corr_s, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1], cbar_kws={'label': 'Spearman rho'})
axes[1].set_title("8.1 (B) Power by Source: Spearman Rank Correlation [GENERAL]", fontweight='bold')

plt.tight_layout()
plt.show()

# 8.2 Power by Source at Day & Night Peaks [PROJECT-BIASED]
fig, axes = plt.subplots(2, 2, figsize=(17, 13))

corr_dp_p = df_lc_day_peak[SOURCES_MW].corr(method='pearson')
corr_dp_s = df_lc_day_peak[SOURCES_MW].corr(method='spearman')
sns.heatmap(corr_dp_p, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0, 0])
axes[0, 0].set_title("8.2 (A) Day Peak: Pearson Linear Correlation [PROJECT-BIASED]", fontweight='bold')
sns.heatmap(corr_dp_s, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0, 1])
axes[0, 1].set_title("8.2 (B) Day Peak: Spearman Rank Correlation [PROJECT-BIASED]", fontweight='bold')

corr_np_p = df_lc_night_peak[SOURCES_MW].corr(method='pearson')
corr_np_s = df_lc_night_peak[SOURCES_MW].corr(method='spearman')
sns.heatmap(corr_np_p, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1, 0])
axes[1, 0].set_title("8.2 (C) Night Peak: Pearson Linear Correlation [PROJECT-BIASED]", fontweight='bold')
sns.heatmap(corr_np_s, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1, 1])
axes[1, 1].set_title("8.2 (D) Night Peak: Spearman Rank Correlation [PROJECT-BIASED]", fontweight='bold')

plt.tight_layout()
plt.show()

# 8.3 Hydro Generation vs. Reservoir Storage & Rainfall Correlations [PROJECT-BIASED]
df_hydro_corr = pd.DataFrame(index=df_ed_pivot.index)
for h in HYDRO_COMPLEXES:
    if h in df_ed_pivot.columns:
        df_hydro_corr[h] = df_ed_pivot[h]

df_hydro_corr['National_Avg_Storage_%'] = df_res_summary['National_Avg_Storage_Pct']
df_hydro_corr['National_Total_Rain_mm'] = df_res_summary['National_Total_Rainfall_mm']
df_hydro_corr = df_hydro_corr.dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
corr_hydro_p = df_hydro_corr.corr(method='pearson')
corr_hydro_s = df_hydro_corr.corr(method='spearman')

sns.heatmap(corr_hydro_p, annot=True, fmt='.2f', cmap='PuBuGn', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("8.3 (A) Hydro Generation vs. Storage & Rain: Pearson [PROJECT-BIASED]", fontweight='bold')

sns.heatmap(corr_hydro_s, annot=True, fmt='.2f', cmap='PuBuGn', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("8.3 (B) Hydro Generation vs. Storage & Rain: Spearman [PROJECT-BIASED]", fontweight='bold')

plt.tight_layout()
plt.show()


---
## Section 9: Engineering Synthesis & Findings `[PROJECT-BIASED]`

### 1. Empirical Evidence on the Three Project Aims

#### Aim 1: Back Pumping (Pumped Storage Hydropower - PSH)
- **Off-Peak Energy Headroom (02:00 – 04:30):**  
  Empirical measurements confirm that the national demand regularly dips below **1,500 MW** during night off-peak troughs, while coal base load operates near **800–850 MW**. This establishes an off-peak pumping headroom of approximately **300–500 MW** where base-load or wind energy can drive pumps without forcing thermal starts.
- **Peak-to-Off-Peak Demand Swing:**  
  The system regularly swings by over **1,200 MW** between off-peak (03:30) and the evening peak (19:00). Storing water overnight to generate 400–600 MW during the 1.5-hour evening peak window (`18:30`–`20:00`) directly shaves the primary grid stress period.

#### Aim 2: Solar-Excess Back Pumping
- **Midday Surplus Reality Check:**  
  Current utility-scale solar dispatch reaches peak output of **~400–550 MW** between 11:30 and 13:30. In the current grid mix, this solar energy is absorbed into daytime demand (commercial and industrial load). True instantaneous **negative net load** (surplus beyond total national demand) does **not** currently occur.
- **Solar-Excess Pumping Feasibility:**  
  Back pumping during solar peak is currently viable as an **energy displacement strategy** rather than a curtailment prevention measure. As planned solar capacity expands beyond 1,500 MW, midday net load will drop below thermal minimums, making dedicated solar pumping mandatory to avoid renewable curtailment.

#### Aim 3: Fossil Fuel Reduction
- **Evening Peaker Displacement:**  
  During the night peak (`18:30`–`20:00`), Thermal-Oil plants (diesel/heavy fuel oil) dispatch between **150 MW and 450 MW** at high operational cost. Pumped hydro capacity of **300–400 MW** discharged strictly during this 90-minute window can displace virtually 100% of expensive oil peaker dispatch.
- **Coal Cycling Mitigation:**  
  By utilizing off-peak pumping, base-load coal units can maintain flat, high-efficiency output overnight rather than throttling down toward minimum stable generation limits.

---

### 2. Methodological Boundaries: What the Data Cannot Show

- **Hydraulic Interconnections:**  
  The dataset records individual reservoir storage percentages but does **not** contain pipe/tunnel flow capacities, hydraulic head, or turbine efficiency curves between specific upper and lower reservoir pairs (e.g., Kothmale–Upper Kothmale or Castlereigh–Maussakalle).
- **Transmission Constraints:**  
  The data represents national aggregate dispatch. Regional transmission bottlenecks (e.g., Northern Province wind evacuation lines) cannot be inferred without bus-level load flow data.
- **Scenarios vs. Evidence:**  
  All figures in Sections 0 through 8 present **strictly empirical evidence**. Potential pumping hours, water volume pumped, and GWh offset represent **prospective what-if scenarios** that require PSH facility rating assumptions.
